In [1]:
using Random
include(joinpath(pwd(), "src/cargo_generation.jl"))
Random.seed!(4242)

trainsize = 20



# 20
seedstrain = [rand(1:10000) for i in 1:trainsize]
cargo_a_90_6 = [genereate_cargo_structs(floor(Int,20),seed = i,num_ports = 6) for i in seedstrain]


20-element Vector{Vector{Cargo}}:
 [Cargo("container", 8, 0.0f0, 1600.0f0), Cargo("machinery", 6, 0.0f0, 1400.0f0), Cargo("machinery", 7, 0.0f0, 1400.0f0), Cargo("machinery", 7, 0.0f0, 1400.0f0), Cargo("container", 8, 0.027790828f0, 1600.0f0), Cargo("container", 8, 2.8993585f0, 1600.0f0), Cargo("truck", 5, 2.917161f0, 1200.0f0), Cargo("container", 8, 4.476365f0, 1600.0f0), Cargo("truck", 5, 4.652919f0, 1200.0f0), Cargo("truck", 4, 4.943524f0, 1200.0f0), Cargo("car", 3, 5.9278426f0, 1000.0f0), Cargo("car", 3, 6.2585807f0, 1000.0f0), Cargo("truck", 4, 6.9686275f0, 1200.0f0), Cargo("car", 4, 8.455237f0, 1000.0f0), Cargo("container", 7, 8.705189f0, 1600.0f0), Cargo("machinery", 6, 9.873898f0, 1400.0f0), Cargo("car", 3, 11.258245f0, 1000.0f0), Cargo("car", 3, 12.055577f0, 1000.0f0), Cargo("car", 4, 14.504829f0, 1000.0f0), Cargo("truck", 4, 15.356773f0, 1200.0f0)]
 [Cargo("machinery", 6, 0.0f0, 1400.0f0), Cargo("truck", 4, 0.0f0, 1200.0f0), Cargo("container", 7, 0.0f0, 1600.0f0), Cargo("car"

In [2]:
# ============================================================
# CELL 2 — Deck A
# ============================================================
w, l  = 7, 10
unava = [[1,10],[2,10],[6,10],[7,10]]
ramp  = [[3,10],[4,10],[5,10]]
deckAstruct = Deck(w, l, unava, ramp)
deckAmat    = create_deck(deckAstruct)

LoadError: UndefVarError: `Deck` not defined

In [ ]:
# ============================================================
# CELL 3 — MIP instance builder
# Takes cargo list from genereate_cargo_structs directly
# Movements match min_shifts.jl:
#   East, West, North-East, South-East, North, South
# Port mapping: heuristic port p → MIP port p-1
# ============================================================
nrows = 7
ncols = 10

function build_mip_from_cargo(cargo_list, h_val;
        pcostshift = 250,
        timecost   = 500/60)

    unava     = Set([(1,10),(2,10),(6,10),(7,10)])
    ramp      = [(3,10),(4,10),(5,10)]
    S         = [(i,j) for i in 1:nrows for j in 1:ncols if !((i,j) in unava)]
    E         = ramp
    S_no_exit = [s for s in S if !(s in E)]

    # heuristic ports are 3..num_ports+2
    # remap: heuristic port p → MIP port p-1
    # loading port = 1, unloading = 2..max_port-1
    max_heur_port = maximum(c.port for c in cargo_list)
    num_ports     = max_heur_port - 2

    # P consecutive: 1=loading, 2..num_ports+1=unloading
    P = collect(1:num_ports+1)
    C = collect(1:length(cargo_list))

    A     = Tuple{Tuple{Int,Int},Tuple{Int,Int}}[]
    Gamma = Dict{Tuple{Tuple{Int,Int},Tuple{Int,Int}},Vector{Tuple{Int,Int}}}()

    for (i,j) in S
        j == ncols && continue

        # --- East ---
        t = (i, j+1)
        if j+1 <= ncols && t in S
            push!(A, ((i,j), t))
            Gamma[((i,j), t)] = Tuple{Int,Int}[]
        end

        # --- West ---
        t = (i, j-1)
        if j-1 >= 1 && t in S && j+1 <= ncols && (i,j+1) in S
            push!(A, ((i,j), t))
            cl = Tuple{Int,Int}[]
            (i,j-1) in S && push!(cl, (i,j-1))
            (i,j+1) in S && push!(cl, (i,j+1))
            Gamma[((i,j), t)] = cl
        end

        # --- North-East ---
        if i > 1 && j+1 <= ncols
            t = (i-1, j+1)
            if t in S && (i-1,j) in S && (i,j+1) in S
                push!(A, ((i,j), t))
                cl = Tuple{Int,Int}[]
                (i-1,j)   in S && push!(cl, (i-1,j))
                (i,  j+1) in S && push!(cl, (i,  j+1))
                Gamma[((i,j), t)] = cl
            end
        end

        # --- South-East ---
        if i < nrows && j+1 <= ncols
            t = (i+1, j+1)
            if t in S && (i+1,j) in S && (i,j+1) in S
                push!(A, ((i,j), t))
                cl = Tuple{Int,Int}[]
                (i+1,j)   in S && push!(cl, (i+1,j))
                (i,  j+1) in S && push!(cl, (i,  j+1))
                Gamma[((i,j), t)] = cl
            end
        end

        # --- South (matches min_shifts.jl) ---
        if i < nrows && j >= 1 && j+1 <= ncols && j-1 >= 1
            t = (i+1, j)
            if t in S && (i,j+1) in S && (i,j-1) in S && (i+1,j-1) in S
                push!(A, ((i,j), t))
                cl = Tuple{Int,Int}[]
                (i+1,j)   in S && push!(cl, (i+1,j))
                (i,  j-1) in S && push!(cl, (i,  j-1))
                (i+1,j-1) in S && push!(cl, (i+1,j-1))
                Gamma[((i,j), t)] = cl
            end
        end

        # --- North (matches min_shifts.jl) ---
        if i > 1 && j >= 1 && j+1 <= ncols && j-1 >= 1
            t = (i-1, j)
            if t in S && (i,j+1) in S && (i,j-1) in S && (i-1,j-1) in S
                push!(A, ((i,j), t))
                cl = Tuple{Int,Int}[]
                (i-1,j)   in S && push!(cl, (i-1,j))
                (i,  j-1) in S && push!(cl, (i,  j-1))
                (i-1,j-1) in S && push!(cl, (i-1,j-1))
                Gamma[((i,j), t)] = cl
            end
        end
    end

    # remap heuristic ports to MIP ports
    L  = Dict(c => 1                           for c in C)
    U  = Dict(c => cargo_list[c].port - 1      for c in C)
    Ac = Dict(c => Float64(cargo_list[c].arr)  for c in C)
    h  = Dict(c => h_val                       for c in C)
    r  = Dict(c => Float64(cargo_list[c].rev)  for c in C)
    R  = Dict(c => 1                           for c in C)

    C_load   = Dict(p => [c for c in C if L[c]==p] for p in P)
    C_unload = Dict(p => [c for c in C if U[c]==p] for p in P)

    M = Float64(length(cargo_list))

    return (S=S, E=E, S_no_exit=S_no_exit, P=P, C=C, A=A, Gamma=Gamma,
            L=L, U=U, Ac=Ac, h=h, r=r, R=R,
            C_load=C_load, C_unload=C_unload,
            C_shift=pcostshift, timecost=timecost, M=M,
            num_ports=num_ports)
end

In [ ]:
# ============================================================
# CELL 4 — Debug instance
# ============================================================
inst = build_mip_from_cargo(cargo_a_20_6[1], 7,
    pcostshift=250, timecost=500/60)
println("Number of slots:    ", length(inst.S))
println("Number of ramp:     ", length(inst.E))
println("Number of cargo:    ", length(inst.C))
println("Ports in P:         ", inst.P)
println("U values:           ", [inst.U[c] for c in inst.C])
println("C_load keys:        ", keys(inst.C_load))
println("C_unload keys:      ", keys(inst.C_unload))
println("C_unload counts:    ", Dict(p=>length(inst.C_unload[p]) for p in inst.P))
println("Number of arcs:     ", length(inst.A))

In [ ]:
# ============================================================
# CELL 5 — MIP solver
# Continuous delay penalty matching obj_func.jl:
#   wcost = max(0, Dep[1] - PWT) * timecost
# ============================================================
function solve_mip_continuous(inst, perfect_wt; verbose=false)
    (;S,E,S_no_exit,P,C,A,Gamma,L,U,Ac,h,r,R,
      C_load,C_unload,C_shift,timecost,M) = inst

    model = Model(HiGHS.Optimizer); set_silent(model)

    @variable(model, y[C], Bin)
    @variable(model, x[C,S], Bin)
    @variable(model, delta[C,S,P], Bin)
    @variable(model, d[C,S,P])
    @variable(model, f[C,A,P] >= 0)
    @variable(model, ST[C,P] >= 0)
    @variable(model, Dep[P] >= 0)
    @variable(model, delay >= 0)

    # flow upper bound
    @constraint(model,[c in C,a in A,p in P],
        f[c,a,p] <= R[c]*y[c])

    # objective — matches obj_func.jl
    @objective(model, Max,
        sum(r[c]*y[c] for c in C)
        - timecost * delay
        - C_shift * sum(delta[c,s,p] for c in C,s in S,p in P))

    # delay = max(0, Dep[first(P)] - perfect_wt)
    @constraint(model, delay >= Dep[first(P)] - perfect_wt)

    # placement
    @constraint(model,[c in C],
        sum(x[c,s] for s in S) == R[c]*y[c])
    @constraint(model,[s in S],
        sum(x[c,s] for c in C) <= 1)
    @constraint(model,[c in C,s in S,p in P],
        delta[c,s,p] <= x[c,s])

    # flow loading
    @constraint(model,[c in C],
        sum(d[c,e,L[c]] for e in E) <= R[c]*y[c])
    @constraint(model,[c in C,s in S_no_exit],
        d[c,s,L[c]] == -x[c,s])

    # flow unloading
    @constraint(model,[c in C],
        sum(d[c,e,U[c]] for e in E) >= -R[c]*y[c])
    @constraint(model,[c in C,s in S_no_exit],
        d[c,s,U[c]] == x[c,s])

    # flow conservation
    @constraint(model,[c in C,p in P,s in S],
        sum(f[c,a,p] for a in A if a[2]==s) -
        sum(f[c,a,p] for a in A if a[1]==s) == d[c,s,p])

    # shift detection — clearance sets
    for p in P, c in C, dc in C
        c==dc && continue
        U[dc]<=p && continue
        for s in S
            flow_in = [a for a in A if a[2]==s]
            cl_arcs = [a for a in A if s in Gamma[a]]
            @constraint(model,
                sum(f[c,a,p] for a in flow_in) +
                sum(f[c,a,p] for a in cl_arcs) <=
                M*(1-x[dc,s]+delta[dc,s,p]))
        end
    end

    # timing constraints
    @constraint(model,[c in C],
        ST[c,L[c]] >= Ac[c] - M*(1-y[c]))
    for p in P, c in union(C_load[p],C_unload[p])
        @constraint(model, Dep[p] >= ST[c,p]+h[c])
    end
    @constraint(model,[p in 1:length(P)-1],
        Dep[P[p+1]] >= Dep[P[p]])

    t_start = time()
    optimize!(model)
    t_mip   = round(time()-t_start, digits=2)
    status  = termination_status(model)

    if status == OPTIMAL
        obj        = objective_value(model)
        n_accepted = sum(value(y[c])>0.5 ? 1 : 0 for c in C)
        n_shifts   = sum(value(delta[c,s,p])>0.5 ? 1 : 0
                         for c in C,s in S,p in P)
        dep1      = value(Dep[first(P)])
        delay_val = value(delay)

        if verbose
            println("\nCargo decisions:")
            for c in C
                if value(y[c])>0.5
                    slots=[s for s in S if value(x[c,s])>0.5]
                    println("  Cargo $c → accepted, slot: $slots, port: $(U[c]), arr: $(Ac[c]), rev: $(r[c])")
                else
                    println("  Cargo $c → rejected, port: $(U[c]), arr: $(Ac[c]), rev: $(r[c])")
                end
            end
            println("\nDeparture times:")
            for p in P
                println("  Port $p: $(round(value(Dep[p]),digits=2)) min")
            end
            println("\nObjective breakdown:")
            rev = sum(r[c] for c in C if value(y[c])>0.5)
            println("  Revenue:    $(round(rev,digits=2))€")
            println("  Delay:      $(round(delay_val,digits=2)) min")
            println("  Delay cost: $(round(timecost*delay_val,digits=2))€")
            println("  Shifts:     $n_shifts")
            println("  Shift cost: $(round(C_shift*n_shifts,digits=2))€")
            println("  Total obj:  $(round(obj,digits=2))€")
        end

        return (obj=obj, n_accepted=n_accepted, n_shifts=n_shifts,
                dep1=dep1, delay=delay_val,
                status=status, time=t_mip), model
    else
        return (obj=nothing, n_accepted=0, n_shifts=0,
                dep1=0.0, delay=0.0,
                status=status, time=t_mip), model
    end
end

In [ ]:
# ============================================================
# CELL 6 — Run MIP verbose on first instance
# ============================================================
inst1 = build_mip_from_cargo(cargo_a_20_6[1], 7,
    pcostshift=250, timecost=500/60)
tmp_deck, tmp_cargo_on = load_random(copy(deckAmat), copy(cargo_a_20_6[1]))
perfect_wt1 = perfect_wait_time(tmp_deck, tmp_cargo_on)
println("Perfect wait time: ", round(perfect_wt1, digits=2), " min")
result1, model1 = solve_mip_continuous(inst1, perfect_wt1, verbose=true)

In [ ]:
# ============================================================
# CELL 7 — Run MIP on all training instances
# ============================================================
function run_mip_on_instances(instances, label;
        h_val=7, verbose=false)
    println("="^90)
    println("MIP — Continuous delay penalty — $label")
    println("Parameters: timecost=500/60 €/min, C_shift=250, handling_time=$(h_val) min")
    println("="^90)
    @printf("%-12s  %10s  %8s  %8s  %10s  %10s  %8s\n",
        "Instance", "MIP Obj", "Acc", "Shifts",
        "Revenue", "Delay cost", "Time (s)")
    println("-"^90)

    obj_vals   = Float64[]
    times      = Float64[]
    acc_vals   = Int[]
    shift_vals = Int[]

    for (i, cargo) in enumerate(instances)
        inst = build_mip_from_cargo(cargo, h_val,
            pcostshift=250, timecost=500/60)
        tmp_deck, tmp_cargo_on = load_random(copy(deckAmat), copy(cargo))
        perfect_wt = perfect_wait_time(tmp_deck, tmp_cargo_on)
        mip_result, _ = solve_mip_continuous(inst, perfect_wt,
                             verbose=verbose)

        if mip_result.obj !== nothing
            rev        = sum(inst.r[c] for c in inst.C)
            delay_cost = round((500/60)*mip_result.delay, digits=2)
            @printf("%-12s  %10.2f  %8d  %8d  %10.2f  %10.2f  %8.2f\n",
                "inst $i",
                mip_result.obj,
                mip_result.n_accepted,
                mip_result.n_shifts,
                rev,
                delay_cost,
                mip_result.time)
            println("  ↳ PWT: $(round(perfect_wt,digits=2)) min  |  Delay: $(round(mip_result.delay,digits=2)) min")
            push!(obj_vals,   mip_result.obj)
            push!(times,      mip_result.time)
            push!(acc_vals,   mip_result.n_accepted)
            push!(shift_vals, mip_result.n_shifts)
        else
            @printf("%-12s  %10s  %8d  %8d  %10s  %10s  %8.2f\n",
                "inst $i", "INFEASIBLE",
                0, 0, "-", "-", mip_result.time)
            push!(obj_vals,   NaN)
            push!(times,      mip_result.time)
            push!(acc_vals,   0)
            push!(shift_vals, 0)
        end
    end

    println("="^90)
    println("Summary:")
    valid = filter(!isnan, obj_vals)
    @printf("  Mean obj:      %.2f\n", isempty(valid) ? 0.0 : mean(valid))
    @printf("  Mean time:     %.2f s\n", mean(times))
    @printf("  Mean accepted: %.1f / %d\n", mean(acc_vals), length(instances[1]))
    @printf("  Mean shifts:   %.2f\n", mean(shift_vals))
    println("="^90)

    return obj_vals, times, acc_vals, shift_vals
end

run_mip_on_instances(cargo_a_20_6, "DeckA 20 cargo 6 ports",
    h_val=7, verbose=false)